<a href="https://colab.research.google.com/github/somendrew/LangGraph_tutorial/blob/main/1_chain_vs_graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
api_key = userdata.get('api_key')

# LangChain Way

In [2]:
!pip install -q langchain_core langchain_openai langgraph typing

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 27.2 MB/s eta 0:00:00


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages([
    'user', "Answer this question :{question} "
])


llm = ChatOpenAI(model = "gpt-4o-mini",api_key = api_key)

chain = prompt | llm

result = chain.invoke({"question": "what is 2+2?"})

print(result.content)

2 + 2 equals 4.


# LangGraph Way

In [6]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from typing import TypedDict



llm = ChatOpenAI(model='gpt-4o-mini',api_key = api_key)

#state: The data flow throught the graph
class State(TypedDict):
  question: str
  answer: str

#node: that take a state and return another state
def call_llm(state:State)-> State:
  response = llm.invoke(state['question'])
  return {"answer" : response.content}

#create graph
graph = StateGraph(State)
graph.add_node("call_llm",call_llm) #add node
graph.set_entry_point("call_llm") #where to start
graph.add_edge("call_llm", END) #Where to end

app = graph.compile()

result = app.invoke({"question": "what is 2+2?","answer": ""})
print(result['answer'])

2 + 2 equals 4.


In [7]:
print(result)

{'question': 'what is 2+2?', 'answer': '2 + 2 equals 4.'}
